<!-- 학습 보강 셀 -->

# 08. VectorStoreIndex 학습 흐름

이 노트북은 문서를 벡터화해 검색 가능한 인덱스로 만드는 가장 핵심적인 예제입니다.
`VectorStoreIndex`는 RAG에서 “질문과 관련된 문서 조각을 찾는 역할”을 담당합니다.

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

In [2]:
# OLLAMA_MODEL_PREP_CELL
# Ollama 모델은 pip/requirements.txt로 설치되지 않습니다.
# 이 셀은 노트북 실행 전에 필요한 로컬 Ollama 모델이 있는지 확인하고, 없으면 자동으로 pull 합니다.
import subprocess

OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_LLM_MODEL = 'gemma2:2b'
OLLAMA_EMBED_MODEL = 'nomic-embed-text'

def _installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = _installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [OLLAMA_LLM_MODEL, OLLAMA_EMBED_MODEL]:
    ensure_ollama_model(model_name)

이미 설치됨: gemma2:2b
이미 설치됨: nomic-embed-text


In [3]:
# LLM과 임베딩 모델 설정
# - LLM은 답변 생성, 임베딩 모델은 문서 검색용 벡터 생성에 사용됩니다.
llm = Ollama(
    model=OLLAMA_LLM_MODEL,
    temperature=0,
    request_timeout=120,
    base_url=OLLAMA_BASE_URL,
)

embed_model = OllamaEmbedding(
    model_name=OLLAMA_EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

In [4]:
# 데이터 로드
# - 아래 질문이 논문 제목을 묻고 있으므로 pdf_sample2의 Attention Is All You Need 논문을 사용합니다.
# - pdf_sample1은 미국 AI 정책 리포트라서 이 노트북의 질문 의도와 맞지 않습니다.
documents = SimpleDirectoryReader('../NewData/pdf_sample2/').load_data()
print('읽어온 문서 수:', len(documents))

읽어온 문서 수: 11


<!-- 학습 보강 셀 -->

## 질문과 데이터의 정합성

이 노트북은 논문 제목을 묻는 질문을 실행하므로, 데이터도 논문 PDF인 `pdf_sample2`를 사용합니다.
질문 의도와 데이터가 맞지 않으면 인덱스가 정상이어도 답변은 부정확해집니다.

In [5]:
# 인덱스 생성 및 데이터 임베딩
# - from_documents는 Document -> Node 분할 -> 임베딩 -> 인덱스 저장을 한 번에 수행합니다.
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    show_progress=True,
)

/Users/cheng80/Documents/WorkSpace/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 12/12 [00:01<00:00,  8.78it/s]


<!-- 학습 보강 셀 -->

## VectorStoreIndex가 검색하는 방식

질문도 임베딩 벡터로 바뀌고, 문서 Node의 임베딩 벡터와 가까운 순서로 검색됩니다.
즉, 키워드가 정확히 일치하지 않아도 의미가 가까우면 검색될 수 있습니다.
이것이 벡터 검색이 단순 문자열 검색과 다른 핵심입니다.

In [6]:
# 쿼리 엔진 생성
query_engine = index.as_query_engine(llm=llm)

2026-06-02 11:31:41,626 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [7]:
# 영어 질문 실행
query = 'what is the title of the paper? let me know the title of the paper'
response = query_engine.query(query)
print(response)

2026-06-02 11:31:41,665 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:31:43,549 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


The title of the paper is "Attention Is All You Need". 



<!-- 학습 보강 셀 -->

## 같은 질문을 다른 언어로 해보기

영어 논문에 영어로 질문한 결과와 한국어로 질문한 결과를 비교하면, 임베딩 모델의 다국어 처리 능력과 LLM의 답변 생성 능력을 함께 관찰할 수 있습니다.
두 결과가 다르면 검색 결과가 달라진 것인지, LLM 답변 생성이 달라진 것인지 source_nodes로 추가 확인할 수 있습니다.

In [8]:
# 한국어 질문 실행
# - 같은 인덱스라도 질문 언어에 따라 검색/답변 품질이 달라질 수 있습니다.
query = '이 논문의 제목이 뭐야? 논문의 제목을 알려줘'
response = query_engine.query(query)
print(response)

2026-06-02 11:31:43,572 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:31:46,174 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


This document's title is "Attention Is All You Need". 

